# DeBERTa-v3-base: дообучение для прогноза рейтинга

Проверяем, помогает ли адаптация предобученного трансформера к отзывам. Сохраняем ту же валидационную часть, что в `modeling.ipynb` и `mpnet_catboost.ipynb`. Чтобы эксперимент поместился на Apple M1 с 8 ГБ памяти, обучаем классификатор и два верхних слоя энкодера, а остальные веса оставляем замороженными. Используем **все 47 980 обучающих отзывов**, одну эпоху и максимум 128 токенов. На случайном образце 5 000 отзывов этот лимит обрезал около 13,7% текстов. Это ограничение эксперимента, а не свойство датасета в целом.

In [1]:
from hashlib import sha256
from pathlib import Path
from time import perf_counter
import os
import random

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import StratifiedGroupKFold
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

seed = 2026
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.set_num_threads(4)

root = Path.cwd()
if not (root / "data" / "raw" / "train.csv").exists():
    root = root.parent
data_path = root / "data" / "raw" / "train.csv"
if not data_path.exists():
    raise FileNotFoundError("Поместите train.csv в data/raw/")
train = pd.read_csv(data_path)
assert list(train.columns) == ["Id", "Review", "Rating"]
clean = train.drop_duplicates(subset=["Review"], keep="first").reset_index(drop=True)
groups = clean["Review"].str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
train_idx, valid_idx = next(splitter.split(clean["Review"], clean["Rating"], groups))
assert set(groups.iloc[train_idx]).isdisjoint(set(groups.iloc[valid_idx]))
y_train = clean["Rating"].iloc[train_idx].to_numpy()
y_valid = clean["Rating"].iloc[valid_idx].to_numpy()
print(f"Обучение: {len(train_idx):,}; валидация: {len(valid_idx):,}")

/Users/vlanuka/Documents/ChatGPT/ncot/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Обучение: 47,980; валидация: 11,996


## Токенизация

Используем исходный SentencePiece-токенизатор DeBERTa. Метки 1–5 преобразуем в классы 0–4 только на время обучения. Проверочные тексты токенизируются теми же правилами, но их оценки не участвуют в обучении.

In [2]:
model_id = "microsoft/deberta-v3-base"
model_revision = "8ccc9b6f36199bec6961081d44eb72fb3f7353f3"
max_length = 128
batch_size = 8
num_epochs = 1
device = "mps" if torch.backends.mps.is_available() else "cpu"
fingerprint = sha256(data_path.read_bytes()).hexdigest()[:12]
cache_dir = root / "data" / "cache"
cache_dir.mkdir(parents=True, exist_ok=True)
model_dir = cache_dir / f"deberta_v3_base_top2_v1_{model_revision[:8]}_{max_length}_{fingerprint}_split2026"

tokenizer = AutoTokenizer.from_pretrained(model_id, revision=model_revision, use_fast=False)
started = perf_counter()
train_tokens = tokenizer(
    clean["Review"].iloc[train_idx].tolist(),
    padding="max_length", truncation=True, max_length=max_length, return_tensors="pt",
)
valid_tokens = tokenizer(
    clean["Review"].iloc[valid_idx].tolist(),
    padding="max_length", truncation=True, max_length=max_length, return_tensors="pt",
)
train_dataset = TensorDataset(
    train_tokens["input_ids"], train_tokens["attention_mask"],
    torch.tensor(y_train - 1, dtype=torch.long),
)
valid_dataset = TensorDataset(
    valid_tokens["input_ids"], valid_tokens["attention_mask"],
    torch.tensor(y_valid - 1, dtype=torch.long),
)
print(f"Токенизация: {(perf_counter() - started) / 60:.1f} мин; устройство: {device}")

Токенизация: 0.2 мин; устройство: mps


## Дообучение

Фиксируем параметры до оценки: cross-entropy для пяти классов, AdamW, линейное снижение скорости обучения, одна эпоха. Обучаем только два верхних слоя энкодера, классификационную голову и pooler. Полученный файл модели хранится локально в `data/cache/`, который исключён из Git.

In [3]:
if model_dir.exists():
    model = AutoModelForSequenceClassification.from_pretrained(model_dir, local_files_only=True)
    print("Использована локальная обученная модель")
else:
    model = AutoModelForSequenceClassification.from_pretrained(
        model_id, revision=model_revision, num_labels=5
    )
    for parameter in model.deberta.parameters():
        parameter.requires_grad = False
    for layer in model.deberta.encoder.layer[-2:]:
        for parameter in layer.parameters():
            parameter.requires_grad = True

    encoder_parameters = [
        parameter for layer in model.deberta.encoder.layer[-2:]
        for parameter in layer.parameters()
    ]
    head_parameters = list(model.pooler.parameters()) + list(model.classifier.parameters())
    trainable_parameters = encoder_parameters + head_parameters
    print("Обучаемых параметров:", f"{sum(p.numel() for p in trainable_parameters):,}")
    model.to(device).train()
    loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        generator=torch.Generator().manual_seed(seed), num_workers=0,
    )
    optimizer = torch.optim.AdamW([
        {"params": encoder_parameters, "lr": 3e-5},
        {"params": head_parameters, "lr": 1e-4},
    ], foreach=False)
    total_steps = len(loader) * num_epochs
    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=round(total_steps * 0.05),
        num_training_steps=total_steps,
    )
    started = perf_counter()
    loss_sum = 0.0
    for epoch in range(num_epochs):
        for step, (input_ids, attention_mask, labels) in enumerate(loader, start=1):
            optimizer.zero_grad(set_to_none=True)
            output = model(
                input_ids=input_ids.to(device),
                attention_mask=attention_mask.to(device),
                labels=labels.to(device),
            )
            loss = output.loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(trainable_parameters, max_norm=1.0)
            optimizer.step()
            scheduler.step()
            loss_sum += loss.item()
            if step % 500 == 0 or step == len(loader):
                print(
                    f"Эпоха {epoch + 1}, шаг {step:,}/{len(loader):,}, "
                    f"средняя loss {loss_sum / step:.4f}, "
                    f"время {(perf_counter() - started) / 60:.1f} мин",
                    flush=True,
                )
    model.eval()
    temporary_dir = model_dir.with_name(model_dir.name + ".partial")
    temporary_dir.mkdir(parents=True, exist_ok=True)
    model.save_pretrained(temporary_dir, safe_serialization=True)
    tokenizer.save_pretrained(temporary_dir)
    os.replace(temporary_dir, model_dir)
    print(f"Обучение: {(perf_counter() - started) / 60:.1f} мин")
model.to(device).eval()

Использована локальная обученная модель


/Users/vlanuka/Documents/ChatGPT/ncot/.venv/lib/python3.12/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


DebertaV2ForSequenceClassification(
  (deberta): DebertaV2Model(
    (embeddings): DebertaV2Embeddings(
      (word_embeddings): Embedding(128100, 768, padding_idx=0)
      (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True, bias=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): DebertaV2Encoder(
      (layer): ModuleList(
        (0-11): 12 x DebertaV2Layer(
          (attention): DebertaV2Attention(
            (self): DisentangledSelfAttention(
              (query_proj): Linear(in_features=768, out_features=768, bias=True)
              (key_proj): Linear(in_features=768, out_features=768, bias=True)
              (value_proj): Linear(in_features=768, out_features=768, bias=True)
              (pos_dropout): Dropout(p=0.1, inplace=False)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): DebertaV2SelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerN

## Оценка

Из вероятностей пяти классов выбираем **медиану распределения**: первую оценку, для которой накопленная вероятность достигла 0,5. Это согласовано с метрикой MAE. Сравниваем с простым бейзлайном на тех же 11 996 проверочных отзывах.

In [4]:
probability_cache = model_dir / "valid_probabilities.npy"
if probability_cache.exists():
    probabilities = np.load(probability_cache)
    print("Использован локальный кэш вероятностей валидации")
else:
    loader = DataLoader(valid_dataset, batch_size=32, shuffle=False, num_workers=0)
    probability_batches = []
    started = perf_counter()
    with torch.inference_mode():
        for step, (input_ids, attention_mask, _) in enumerate(loader, start=1):
            logits = model(
                input_ids=input_ids.to(device),
                attention_mask=attention_mask.to(device),
            ).logits
            probability_batches.append(torch.softmax(logits, dim=1).cpu().numpy())
            if step % 100 == 0 or step == len(loader):
                print(f"Проверено {min(step * 32, len(valid_dataset)):,} / {len(valid_dataset):,}", flush=True)
    probabilities = np.concatenate(probability_batches)
    np.save(probability_cache, probabilities)
    print(f"Время проверки: {(perf_counter() - started) / 60:.1f} мин")
assert probabilities.shape == (len(y_valid), 5)
predictions = np.argmax(np.cumsum(probabilities, axis=1) >= 0.5, axis=1) + 1
mae = mean_absolute_error(y_valid, predictions)
print(f"MAE DeBERTa-v3-base (два верхних слоя): {mae:.5f}")

by_rating = pd.DataFrame({
    "строк": [int((y_valid == rating).sum()) for rating in range(1, 6)],
    "MAE": [
        mean_absolute_error(y_valid[y_valid == rating], predictions[y_valid == rating])
        for rating in range(1, 6)
    ],
}, index=pd.Index(range(1, 6), name="Rating"))
display(by_rating.round(4))

Проверено 3,200 / 11,996


Проверено 6,400 / 11,996


Проверено 9,600 / 11,996


Проверено 11,996 / 11,996


Время проверки: 11.5 мин


MAE DeBERTa-v3-base (два верхних слоя): 0.14722


,строк,MAE
Rating,,
1,3732,0.0654
2,326,0.9233
3,336,1.1220
4,670,0.7687
5,6932,0.0475


## Выводы

- Частично дообученная DeBERTa-v3-base дала **MAE 0,14722** на той же валидационной части, где TF-IDF + Logistic Regression дала **0,19165**. Улучшение — **0,04443 MAE** (около 23,2%). По сравнению с MPNet + CatBoost (**0,44325**) результат также существенно лучше.
- По оценкам 1 и 5 MAE составила 0,0654 и 0,0475. Для редких оценок 2–4 ошибки всё ещё заметно выше: 0,9233 / 1,1220 / 0,7687. Общая метрика не означает, что промежуточные оценки решены хорошо.
- Обучение двух верхних слоёв и классификатора на всех 47 980 обучающих отзывах заняло 55,7 мин на Apple M1; проверка 11 996 отзывов — 11,1 мин. Вход ограничен 128 токенами, поэтому часть длинных отзывов обрезается. Полное дообучение всех слоёв и другие настройки в этом эксперименте не проверялись.
- При выборе модели для Docker-сервиса отдельно проверим задержку одиночного предсказания на CPU: лучшая MAE сама по себе не гарантирует выполнение требования по скорости.